In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix
from xgboost import XGBClassifier
from sklearn.model_selection import KFold


In [41]:
df_1_1 = pd.read_csv("mini_1.csv")
df_1_5 = pd.read_csv("mini_5.csv")


In [27]:
def evaluate_model(X, y, model, n_runs=10):
    f1_scores = []
    conf_matrices = []

    for i in range(n_runs):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.3,
            random_state=i,
            stratify=y
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # F1
        f1_scores.append(f1_score(y_test, y_pred))

        # Confusion matrix
        conf_matrices.append(confusion_matrix(y_test, y_pred))

    # Convert to numpy array for averaging
    conf_matrices = np.array(conf_matrices)

    return (
        np.mean(f1_scores),
        np.std(f1_scores),
        np.mean(conf_matrices, axis=0)   
    )


In [19]:

def in_distribution_hashtags(df, n_runs=10):
    print("In-Distribution")
    results = {}
    model = XGBClassifier(
                max_depth=8,
                learning_rate=0.3,
                n_estimators=100,
                scale_pos_weight=3,
                min_child_weight=1,
                subsample=1.0,
                objective="binary:logistic",
                random_state=42,
                eval_metric="logloss",
                tree_method="hist"
             )
    
    for tag in df["hashtag"].unique():

        df_tag = df[df["hashtag"] == tag]


        X = df_tag.drop(columns=["label", "hashtag", "A_id", "S_id", "P_id"])
        y = df_tag["label"]

        mean_f1, std_f1, avg_conf = evaluate_model(X, y, model, n_runs=n_runs)

        results[tag] = (mean_f1, std_f1)

        print(f"Hashtag: {tag}")
        print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")
        print("-" * 40)



In [20]:


def out_of_distribution_hashtags(df):

    hashtags = df["hashtag"].unique()
    print("Out-of-Distribution")
    results = {}

    model = XGBClassifier(
                max_depth=8,
                learning_rate=0.3,
                n_estimators=100,
                scale_pos_weight=3,
                min_child_weight=1,
                subsample=1.0,
                objective="binary:logistic",
                random_state=42,
                eval_metric="logloss",
                tree_method="hist"
             )
    
    for tag in hashtags:

        test_df = df[df["hashtag"] == tag].copy()
        train_df = df[df["hashtag"] != tag].copy()
        print(len(train_df), len(test_df))

        X_test = test_df.drop(columns=["label", "hashtag", "A_id", "S_id", "P_id"])
        y_test = test_df["label"]

        f1_scores = []

        kf = KFold(n_splits=10, shuffle=True, random_state=42)

        for train_index, _ in kf.split(train_df):

            train_subset = train_df.iloc[train_index]

            X_train = train_subset.drop(columns=["label", "hashtag", "A_id", "S_id", "P_id"])
            y_train = train_subset["label"]

            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
            f1_scores.append(f1_score(y_test, y_pred))

        mean_f1 = np.mean(f1_scores)
        std_f1 = np.std(f1_scores)

        print(f"Hashtag: {tag}")
        print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")
        print("-" * 40)

        results[tag] = (mean_f1, std_f1)





In [47]:
out_of_distribution_hashtags(df_1_1)

Out-of-Distribution
65222 2747
Hashtag: TheTraitors
F1: 0.7579 ± 0.0041
----------------------------------------
58364 9605
Hashtag: Anime
F1: 0.7471 ± 0.0028
----------------------------------------
56203 11766
Hashtag: art
F1: 0.7615 ± 0.0017
----------------------------------------
65024 2945
Hashtag: olympics
F1: 0.7519 ± 0.0033
----------------------------------------
62688 5281
Hashtag: AI
F1: 0.7589 ± 0.0028
----------------------------------------
59159 8810
Hashtag: booksky
F1: 0.7571 ± 0.0016
----------------------------------------
58813 9156


KeyboardInterrupt: 

In [46]:
in_distribution_hashtags(df_1_1)

In-Distribution
Hashtag: TheTraitors
F1: 0.7293 ± 0.0089
----------------------------------------
Hashtag: Anime
F1: 0.7369 ± 0.0074
----------------------------------------
Hashtag: art
F1: 0.7528 ± 0.0062
----------------------------------------
Hashtag: olympics
F1: 0.7288 ± 0.0104
----------------------------------------
Hashtag: AI
F1: 0.7367 ± 0.0053
----------------------------------------
Hashtag: booksky
F1: 0.7431 ± 0.0091
----------------------------------------
Hashtag: Pokemon
F1: 0.7503 ± 0.0041
----------------------------------------
Hashtag: superbowl
F1: 0.7349 ± 0.0085
----------------------------------------
Hashtag: ICE
F1: 0.7437 ± 0.0078
----------------------------------------
Hashtag: gaza
F1: 0.7378 ± 0.0103
----------------------------------------


In [81]:
#mixed 
model = XGBClassifier(
    max_depth=8,
    learning_rate=0.3,
    n_estimators=100,
    scale_pos_weight=1,
    min_child_weight=1,
    subsample=1.0,
    reg_lambda=1.0,
    reg_alpha=0.5,
    objective="binary:logistic",
    random_state=42,
    eval_metric="logloss",
    tree_method="hist"
)

id_cols = ["A_id", "S_id", "P_id", "hashtag"]

X = df_1_5.drop(columns=id_cols + ["label"])
y = df_1_5["label"]
X = X.fillna(0)


mean_f1, std_f1, avg_conf = evaluate_model(X, y, model, n_runs=10)
print(f"Mixed 1:5:")
print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")
print("-" * 40)


# X = df_1_1.drop(columns=id_cols + ["label"])
# y = df_1_1["label"]
# X = X.fillna(0)


# mean_f1, std_f1, avg_conf = evaluate_model(X, y, model, n_runs=10)
# print(f"Mixed 1:1:")
# print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")
# print("-" * 40)


Mixed 1:5:
F1: 0.4300 ± 0.0026
----------------------------------------


In [82]:

booster = model.get_booster()

# Get raw gain importance
importance_dict = booster.get_score(importance_type="gain")

# Convert to DataFrame
importance_df = pd.DataFrame(
    list(importance_dict.items()),
    columns=["feature", "gain"]
)

# Sort by gain
importance_df = importance_df.sort_values(
    by="gain",
    ascending=False
).reset_index(drop=True)

# Convert gain to percentage importance
total_gain = importance_df["gain"].sum()
importance_df["gain"] = (
    importance_df["gain"] / total_gain
)

print(importance_df)

                   feature      gain
0    U-HA_R_RetweetPercent  0.249407
1   U-HA_R_AverageInterval  0.222103
2        U-P_R_TweetNumDay  0.036385
3     U-P_R_FolloweeNumDay  0.035480
4         U-P_R_ProfileUrl  0.032250
5         U-P_R_AccountAge  0.031368
6        U-P_R_FollowerNum  0.030882
7           U-P_R_TweetNum  0.030372
8     U-HA_R_RetweetedRate  0.029848
9        U-P_R_FolloweeNum  0.028991
10    U-P_R_FollowerNumDay  0.028912
11    U-P_S_FolloweeNumDay  0.019454
12       U-P_S_FollowerNum  0.019149
13    U-P_S_FollowerNumDay  0.018754
14       U-P_S_TweetNumDay  0.018702
15       U-P_S_FolloweeNum  0.018631
16          U-P_S_TweetNum  0.018518
17    U-HA_S_RetweetedRate  0.018414
18    U-P_SR_followersDiff  0.018413
19  U-HA_S_AverageInterval  0.018297
20        U-P_S_AccountAge  0.018154
21   U-HA_S_RetweetPercent  0.017659
22        U-P_S_ProfileUrl  0.016962
23     U-P_R_activeBeforeP  0.013622
24           U-P_R_FollowS  0.009273
